In [ ]:
!pip install tracksdata zarr pyscipopt donfig polars
!git clone https://github.com/royerlab/kaggle-cell-tracking-competition repo

In [ ]:
import os
import json

COMP_DIR_CANDIDATES = [
    "/kaggle/input/competitions/biohub-cell-tracking-during-development",
    "/kaggle/input/biohub-cell-tracking-during-development"
]
COMP_DIR = next(d for d in COMP_DIR_CANDIDATES if os.path.exists(d))
TRAIN_DIR = f"{COMP_DIR}/train"

train_stems = sorted([f[:-5] for f in os.listdir(TRAIN_DIR) if f.endswith(".zarr")])

# Reserve last 4 for validation (expected as 'test' key in this codebase), rest for training
train_vids = train_stems[:-4]
val_vids = train_stems[-4:]

print(f"Training on {len(train_vids)} videos, validating on {len(val_vids)} videos")

with open("repo/splits.json", "w") as f:
    json.dump([{"split": 0, "train": train_vids, "test": val_vids}], f)


In [ ]:
import subprocess
import os

cmd = [
    "python", "scripts/train_unet_transformer.py",
    "--data-dir", TRAIN_DIR,
    "--splits", "splits.json",
    "--split", "0",
    "--epochs", "50",
    "--batch-size", "4",
    "--num-workers", "2"
]

print(" ".join(cmd))
subprocess.run(cmd, cwd="repo", env={**os.environ, "PYTHONPATH": "src"}, check=True)


In [ ]:
import shutil
import os

src_weights = "repo/weights/unet_transformer/split_0/edge_predictor_best.pth"
src_config = "repo/weights/unet_transformer/split_0/config.json"

if os.path.exists(src_weights):
    shutil.copy(src_weights, "/kaggle/working/edge_predictor_best.pth")
    print("Successfully saved edge_predictor_best.pth to working directory.")
if os.path.exists(src_config):
    shutil.copy(src_config, "/kaggle/working/config.json")
    print("Successfully saved config.json to working directory.")
